# 🔐 Post-Quantum Cryptography with liboqs
### Exploring Quantum-Safe algorithms using Open Quantum Safe (OQS) library
- **Library**: liboqs-python
- **Purpose**: Learn and test Post-Quantum Cryptographic algorithms
- **Date**: June 2026

In [1]:
#Import and verify liboqs

import oqs
import os
import hashlib
import json
import cryptography
import struct

print("✅ liboqs imported successfully!")

print(f"📦 Total KEM algorithms available : {len(oqs.get_enabled_kem_mechanisms())}")
print(f"📦 Total SIG algorithms available : {len(oqs.get_enabled_sig_mechanisms())}")
print(f"🔐 liboqs is ready to use!")



liboqs-python faulthandler is disabled
✅ liboqs imported successfully!
📦 Total KEM algorithms available : 29
📦 Total SIG algorithms available : 221
🔐 liboqs is ready to use!


In [2]:
#List all available algorithms

kem_list = oqs.get_enabled_kem_mechanisms()
sig_list = oqs.get_enabled_sig_mechanisms()

print("Total KEM Algorithms: ", len(kem_list))
print("Total SIG Algorithms: ", len(sig_list))

print("\n KEM Algorithms: ")
for i, kem in enumerate(kem_list,1):
    print(f" {i:02}.{kem}")

print("\n Signature Algorithms: ")
for i, sig in enumerate(sig_list,1):
    print(f" {i:02}.{sig}")

Total KEM Algorithms:  29
Total SIG Algorithms:  221

 KEM Algorithms: 
 01.Classic-McEliece-348864
 02.Classic-McEliece-348864f
 03.Classic-McEliece-460896
 04.Classic-McEliece-460896f
 05.Classic-McEliece-6688128
 06.Classic-McEliece-6688128f
 07.Classic-McEliece-6960119
 08.Classic-McEliece-6960119f
 09.Classic-McEliece-8192128
 10.Classic-McEliece-8192128f
 11.Kyber512
 12.Kyber768
 13.Kyber1024
 14.ML-KEM-512
 15.ML-KEM-768
 16.ML-KEM-1024
 17.NTRU-HPS-2048-509
 18.NTRU-HPS-2048-677
 19.NTRU-HPS-4096-821
 20.NTRU-HPS-4096-1229
 21.NTRU-HRSS-701
 22.NTRU-HRSS-1373
 23.sntrup761
 24.FrodoKEM-640-AES
 25.FrodoKEM-640-SHAKE
 26.FrodoKEM-976-AES
 27.FrodoKEM-976-SHAKE
 28.FrodoKEM-1344-AES
 29.FrodoKEM-1344-SHAKE

 Signature Algorithms: 
 01.ML-DSA-44
 02.ML-DSA-65
 03.ML-DSA-87
 04.Falcon-512
 05.Falcon-1024
 06.Falcon-padded-512
 07.Falcon-padded-1024
 08.SPHINCS+-SHA2-128f-simple
 09.SPHINCS+-SHA2-128s-simple
 10.SPHINCS+-SHA2-192f-simple
 11.SPHINCS+-SHA2-192s-simple
 12.SPHINCS+-S

# Kyber(ML-KEM)
ML-KEM allows two parties to securely establish a shared secret without relying on quantum-vulnerable algorithms such as RSA or traditional Diffie–Hellman.

In [3]:
#Explore details of each KEM Algorithm

print(f"{'Algorithm':<35} {'PubKey':>8} {'SecKey':>8} {'Cipher':>8} {'Secret':>8}")
print('-'*80)

for algo in oqs.get_enabled_kem_mechanisms():
    with oqs.KeyEncapsulation(algo) as kem:
        d=kem.details
        print(f"{d['name']:<35} "
              f"{d['length_public_key']:>8} "
              f"{d['length_secret_key']:>8} "
              f"{d['length_ciphertext']:>8} "
              f"{d['length_shared_secret']:>8} "
              f"{d['claimed_nist_level']:>6}")

Algorithm                             PubKey   SecKey   Cipher   Secret
--------------------------------------------------------------------------------
Classic-McEliece-348864               261120     6492       96       32      1
Classic-McEliece-348864f              261120     6492       96       32      1
Classic-McEliece-460896               524160    13608      156       32      3
Classic-McEliece-460896f              524160    13608      156       32      3
Classic-McEliece-6688128             1044992    13932      208       32      5
Classic-McEliece-6688128f            1044992    13932      208       32      5
Classic-McEliece-6960119             1047319    13948      194       32      5
Classic-McEliece-6960119f            1047319    13948      194       32      5
Classic-McEliece-8192128             1357824    14120      208       32      5
Classic-McEliece-8192128f            1357824    14120      208       32      5
Kyber512                                 800     1632    

In [4]:
# Kem - Key Exchange Demo

algorithms = ["ML-KEM-512", "ML-KEM-768", "ML-KEM-1024", "Kyber512", "FrodoKEM-640-AES"]

for algo in algorithms:
    with oqs.KeyEncapsulation(algo) as kem:
        #Generate Key pair
        public_key = kem.generate_keypair()

        #Encapsulate
        ciphertext, shared_secret_sender = kem.encap_secret(public_key)

        #Decapsulate
        shared_secret_receiver = kem.decap_secret(ciphertext)

        match = shared_secret_sender == shared_secret_receiver

        print(f"   Algorithm     : {algo}")
        print(f"   Public Key    : {len(public_key)} bytes")
        print(f"   Ciphertext    : {len(ciphertext)} bytes")
        print(f"   Shared Secret : {len(shared_secret_sender)} bytes")
        print(f"   Secrets Match : {'YES' if match else 'NO'}")
        print()

   Algorithm     : ML-KEM-512
   Public Key    : 800 bytes
   Ciphertext    : 768 bytes
   Shared Secret : 32 bytes
   Secrets Match : YES

   Algorithm     : ML-KEM-768
   Public Key    : 1184 bytes
   Ciphertext    : 1088 bytes
   Shared Secret : 32 bytes
   Secrets Match : YES

   Algorithm     : ML-KEM-1024
   Public Key    : 1568 bytes
   Ciphertext    : 1568 bytes
   Shared Secret : 32 bytes
   Secrets Match : YES

   Algorithm     : Kyber512
   Public Key    : 800 bytes
   Ciphertext    : 768 bytes
   Shared Secret : 32 bytes
   Secrets Match : YES

   Algorithm     : FrodoKEM-640-AES
   Public Key    : 9616 bytes
   Ciphertext    : 9720 bytes
   Shared Secret : 16 bytes
   Secrets Match : YES



In [5]:
#Performance Benchmark - Speed Test

import time

algorithms = ["ML-KEM-512", "ML-KEM-768", "ML-KEM-1024", "Kyber512", "FrodoKEM-640-AES"]

print(f"{'Algorithm':<25} {'KeyGen':>10} {'Encap':>10} {'Decap':>10} {'Total':>10}")
print("-" * 65)

ITERATIONS = 1000

for algo in algorithms:
    try:
        keygen_times = []
        encap_times = []
        decap_times = []

        for _ in range(ITERATIONS):
            with oqs.KeyEncapsulation(algo) as kem:
                #KeyGen
                t0 = time.perf_counter()
                pub = kem.generate_keypair()
                keygen_times.append(time.perf_counter() - t0)

                #Encap
                t0 = time.perf_counter()
                ct,ss1 = kem.encap_secret(pub)
                encap_times.append(time.perf_counter() - t0)

                #decap
                t0 = time.perf_counter()
                ss2 = kem.decap_secret(ct)
                decap_times.append(time.perf_counter() - t0)

        avg_kg = sum(keygen_times) / ITERATIONS * 1000
        avg_en = sum(encap_times)  / ITERATIONS * 1000
        avg_de = sum(decap_times)  / ITERATIONS * 1000
        total  = avg_kg + avg_en + avg_de

        print(f"{algo:<25} {avg_kg:>9.3f}ms {avg_en:>9.3f}ms {avg_de:>9.3f}ms {total:>9.3f}ms")

    except Exception as e:
        print(f"{algo:<25}  Not available")

Algorithm                     KeyGen      Encap      Decap      Total
-----------------------------------------------------------------
ML-KEM-512                    0.328ms     0.400ms     0.508ms     1.235ms
ML-KEM-768                    0.341ms     0.384ms     0.467ms     1.192ms
ML-KEM-1024                   0.495ms     0.541ms     0.642ms     1.679ms
Kyber512                      0.175ms     0.232ms     0.247ms     0.654ms
FrodoKEM-640-AES              6.974ms     6.495ms     6.440ms    19.909ms


# Dilithium (ML-DSA)
ML-DSA allows someone to digitally sign data and others to verify its authenticity using a signature scheme designed to withstand attacks from both classical and quantum computers.

In [6]:
#Digital Signature - Sign and Verify Demo

sig_algorithms = ['ML-DSA-44', 'ML-DSA-65', 'ML-DSA-87']

keygen_times = []
sign_times = []
verify_times = []
ITERATIONS = 1000

message = b"Hello Post Quantum World! This message is quantum safe-signed!"

for algo in sig_algorithms:
    try:
        for _ in range(ITERATIONS):
            with oqs.Signature(algo) as signer:
                #Generate Key Pair
                t0 = time.perf_counter()
                public_key = signer.generate_keypair()
                keygen_times.append(time.perf_counter() - t0)
                
    
                #Sign
                t0 = time.perf_counter()
                signature = signer.sign(message)
                sign_times.append(time.perf_counter() - t0)
    
                #Verify
                t0 = time.perf_counter()
                is_valid = signer.verify(message, signature, public_key)
                verify_times.append(time.perf_counter() - t0)
    
                avg_kg = sum(keygen_times) / ITERATIONS * 1000
                avg_en = sum(sign_times)  / ITERATIONS * 1000
                avg_de = sum(verify_times)  / ITERATIONS * 1000
                total  = avg_kg + avg_en + avg_de

        print(f"   Algorithm   : {algo}")
        print(f"   Public Key  : {len(public_key)} bytes")
        print(f"   Signature   : {len(signature)} bytes")
        print(f"   Valid       : {'YES' if is_valid else 'NO'}")
        print()
        print(f"{'Algorithm':<25} {'KeyGen':>10} {'Signing':>10} {'Verification':>10} {'Total':>10}")
        print("-" * 65)
        print(f"{algo:<25} {avg_kg:>9.3f}ms {avg_en:>9.3f}ms {avg_de:>9.3f}ms {total:>9.3f}ms")
        print()
        print()
            
    except Exception as e:
        print(f"  {algo} not available: {e}\n")

   Algorithm   : ML-DSA-44
   Public Key  : 1312 bytes
   Signature   : 2420 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
ML-DSA-44                     0.567ms     1.808ms     0.599ms     2.974ms


   Algorithm   : ML-DSA-65
   Public Key  : 1952 bytes
   Signature   : 3309 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
ML-DSA-65                     1.502ms     4.421ms     1.516ms     7.440ms


   Algorithm   : ML-DSA-87
   Public Key  : 2592 bytes
   Signature   : 4627 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
ML-DSA-87                     3.015ms     7.820ms     3.064ms    13.899ms




In [7]:
#Tamper Detection Test

with oqs.Signature("ML-DSA-65") as signer:
    public_key = signer.generate_keypair()

    original_message = b"Original Secure Message!!!"
    tampered_message = b"Tampered Secure Message!!!"

    #Sign original
    signature = signer.sign(original_message)

    #Verify Original
    valid_original = signer.verify(original_message, signature, public_key)

    #Verify Tampered
    valid_tampered = signer.verify(tampered_message, signature, public_key)

    print("   Tamper Detection Test")
    print(f"   Original Message Verified : {'PASS' if valid_original else 'FAIL'}")
    print(f"   Tampered Message Verified : {'PASS' if valid_tampered else 'FAIL'}")
    print()
    print("Tamper was detected!" if not valid_tampered else "Tamper NOT detected!")

   Tamper Detection Test
   Original Message Verified : PASS
   Tampered Message Verified : FAIL

Tamper was detected!


# SLH-DSA
SLH-DSA provides quantum-resistant digital signatures using hash-based cryptography, making it a fundamentally different alternative to lattice-based signatures such as ML-DSA.

In [8]:
print("=" * 60)
print("         PHASE 1 — SETUP & KEY GENERATION")
print("=" * 60)

# ── SLH-DSA Algorithm Variant ─────────────────────────────────
# Available variants:
#   SPHINCS+-SHA2-128f-simple   → Fast,  128-bit security
#   SPHINCS+-SHA2-192f-simple   → Fast,  192-bit security
#   SPHINCS+-SHA2-256f-simple   → Fast,  256-bit security (FIPS 205)
#   SPHINCS+-SHA2-128s-simple   → Small, 128-bit security
#   SPHINCS+-SHA2-256s-simple   → Small, 256-bit security

ALGORITHM = "SPHINCS+-SHA2-256f-simple"  #  FIPS 205 compliant variant

#Generate SLH keypair
with oqs.Signature(ALGORITHM) as signer_setup:
    verification_key = signer_setup.generate_keypair()
    signing_key      = signer_setup.export_secret_key()

print(f"\n Algorithm            : {ALGORITHM}")
print(f" Keypair Generated!")
print(f"\n    Verification Key  : {verification_key.hex()[:40]}...")
print(f"    Signing Key       : {signing_key.hex()[:40]}...")
print(f"\n    Verification Key Size : {len(verification_key)} bytes")
print(f"    Signing Key Size      : {len(signing_key)} bytes")
print("\n Setup Complete!\n")

         PHASE 1 — SETUP & KEY GENERATION

 Algorithm            : SPHINCS+-SHA2-256f-simple
 Keypair Generated!

    Verification Key  : cd35955e0fe7f31dd43f5aa0de7ccbee5292671c...
    Signing Key       : e77476fde98aa97c19d73da6111ee9a1d57c8d13...

    Verification Key Size : 64 bytes
    Signing Key Size      : 128 bytes

 Setup Complete!



In [9]:
print("=" * 60)
print("           PHASE 2 — SIGN THE MESSAGE")
print("=" * 60)

# ── Message to Sign ────────────────────────────────────────────

message = b"Hello! This is quantum safe message signed with SLH-DSA(FIPS 205)!!!"
print(f"\nOriginal Message     : {message.decode()}")

# ── Compute Message Hash (optional but good practice) ─────────
message_hash = hashlib.sha256(message).hexdigest()
print(f"Message SHA-256      : {message_hash[:40]}...")

# ── Sign the Message using Signing Key ────────────────────────
with oqs.Signature(ALGORITHM, signing_key) as signer:
    signature = signer.sign(message)

print(f"\n  Signature Generated!")
print(f"   Signature            : {signature.hex()[:40]}...")
print(f"   Signature Size       : {len(signature)} bytes")
print("\n Signing Complete!\n")

           PHASE 2 — SIGN THE MESSAGE

Original Message     : Hello! This is quantum safe message signed with SLH-DSA(FIPS 205)!!!
Message SHA-256      : 3c2bfadb6bddd215daa7c9f133f327f518487f66...

  Signature Generated!
   Signature            : 50bd89154b6152010754bec3256ee2e3e00b135b...
   Signature Size       : 49856 bytes

 Signing Complete!



In [10]:
print("=" * 60)
print("         PHASE 3 — VERIFY THE SIGNATURE")
print("=" * 60)

# ── Verify using Verification Key ─────────────────────────────
with oqs.Signature(ALGORITHM) as verifier:
    is_valid = verifier.verify(
        message,
        signature,
        verification_key       #Public verification key
    )

print(f"\n Verifying Signature...")
print(f"   Verified with        : Verification Key (Public)")
print(f"   Result               : {' VALID' if is_valid else ' INVALID'}")

if not is_valid:
    raise Exception(" Signature verification failed!")

         PHASE 3 — VERIFY THE SIGNATURE

 Verifying Signature...
   Verified with        : Verification Key (Public)
   Result               :  VALID


In [11]:
print("=" * 60)
print("         PHASE 4 — TAMPER DETECTION TESTS")
print("=" * 60)

# ── Test 1: Tampered Message ───────────────────────────────────
tampered_message = b"This message was TAMPERED by an attacker!"

with oqs.Signature(ALGORITHM) as verifier:
    test1 = verifier.verify(tampered_message, signature, verification_key)

print(f"\n    Test 1 — Tampered Message")
print(f"      Original  : {message.decode()[:45]}...")
print(f"      Tampered  : {tampered_message.decode()}")
print(f"      Valid     : {test1}")
print(f"      {' Rejected!' if not test1 else ' Accepted — unexpected!'}")

# ── Test 2: Forged Signature ───────────────────────────────────
forged_sig = os.urandom(len(signature))

with oqs.Signature(ALGORITHM) as verifier:
    test2 = verifier.verify(message, forged_sig, verification_key)

print(f"\n    Test 2 — Forged Signature")
print(f"      Forged Sig : {forged_sig.hex()[:40]}...")
print(f"      Valid      : {test2}")
print(f"      {' Rejected!' if not test2 else ' Accepted — unexpected!'}")

# ── Test 3: Wrong Verification Key ────────────────────────────
with oqs.Signature(ALGORITHM) as fake_setup:
    fake_vk = fake_setup.generate_keypair()

with oqs.Signature(ALGORITHM) as verifier:
    test3 = verifier.verify(message, signature, fake_vk)

print(f"\n    Test 3 — Wrong Verification Key")
print(f"      Fake VK    : {fake_vk.hex()[:40]}...")
print(f"      Valid      : {test3}")
print(f"      {' Rejected!' if not test3 else ' Accepted — unexpected!'}")

print("\n Tamper Detection Tests Complete!\n")


         PHASE 4 — TAMPER DETECTION TESTS

    Test 1 — Tampered Message
      Original  : Hello! This is quantum safe message signed wi...
      Tampered  : This message was TAMPERED by an attacker!
      Valid     : False
       Rejected!

    Test 2 — Forged Signature
      Forged Sig : caca5134265af1a18976d295df452e61b614b007...
      Valid      : False
       Rejected!

    Test 3 — Wrong Verification Key
      Fake VK    : 19b3a7e0c7fba609dc6ce4bbda98c415ae3b3052...
      Valid      : False
       Rejected!

 Tamper Detection Tests Complete!



In [12]:
print("=" * 60)
print("          📊  PHASE 5 — SUMMARY REPORT")
print("=" * 60)

print(f"""
┌──────────────────────────────────────────────────────────┐
│            SLH-DSA Hypertree — Summary Report            │
├─────────────────────────────┬────────────────────────────┤
│ Algorithm                   │ {ALGORITHM}  │
│ NIST Standard               │ FIPS 205                   │
│ Internal Structure          │ Hypertree (d=22 layers)    │
│ Hash Function               │ SHA-2                      │
│ Signing Mode                │ Fast (f)                   │
│ Quantum Safe                │ Yes                        │
├─────────────────────────────┼────────────────────────────┤
│ Public Size                 │ {len(verification_key)} bytes                   │
│ Private Size                │ {len(signing_key)} bytes                  │
│ Signature Size              │ {len(signature)} bytes                │
├─────────────────────────────┼────────────────────────────┤
│ Signature Valid             │ {is_valid}                       │
│ Tampered Message Rejected   │ {not test1}                       │
│ Forged Signature Rejected   │ {not test2}                       │
│ Wrong Key Rejected          │ {not test3}                       │
└─────────────────────────────┴────────────────────────────┘
""")


          📊  PHASE 5 — SUMMARY REPORT

┌──────────────────────────────────────────────────────────┐
│            SLH-DSA Hypertree — Summary Report            │
├─────────────────────────────┬────────────────────────────┤
│ Algorithm                   │ SPHINCS+-SHA2-256f-simple  │
│ NIST Standard               │ FIPS 205                   │
│ Internal Structure          │ Hypertree (d=22 layers)    │
│ Hash Function               │ SHA-2                      │
│ Signing Mode                │ Fast (f)                   │
│ Quantum Safe                │ Yes                        │
├─────────────────────────────┼────────────────────────────┤
│ Public Size                 │ 64 bytes                   │
│ Private Size                │ 128 bytes                  │
│ Signature Size              │ 49856 bytes                │
├─────────────────────────────┼────────────────────────────┤
│ Signature Valid             │ True                       │
│ Tampered Message Rejected   │ True          

# SPHINCS+
SPHINCS+ provides quantum-resistant digital signatures using hash functions, offering a fundamentally different security approach from lattice-based schemes such as ML-DSA.

In [13]:
#Digital Signature - Sign and Verify Demo

sig_algorithms = ['SPHINCS+-SHA2-128f-simple', 'SPHINCS+-SHA2-192f-simple', 'SPHINCS+-SHA2-256f-simple', 'SPHINCS+-SHA2-128s-simple', 'SPHINCS+-SHA2-256s-simple']

keygen_times = []
sign_times = []
verify_times = []
ITERATIONS = 20

message = b"Hello Post Quantum World! This message is quantum safe-signed!"

for algo in sig_algorithms:
    try:
        for _ in range(ITERATIONS):
            with oqs.Signature(algo) as signer:
                #Generate Key Pair
                t0 = time.perf_counter()
                public_key = signer.generate_keypair()
                keygen_times.append(time.perf_counter() - t0)
                
    
                #Sign
                t0 = time.perf_counter()
                signature = signer.sign(message)
                sign_times.append(time.perf_counter() - t0)
    
                #Verify
                t0 = time.perf_counter()
                is_valid = signer.verify(message, signature, public_key)
                verify_times.append(time.perf_counter() - t0)
    
        avg_kg = sum(keygen_times) / ITERATIONS * 1000
        avg_en = sum(sign_times)  / ITERATIONS * 1000
        avg_de = sum(verify_times)  / ITERATIONS * 1000
        total  = avg_kg + avg_en + avg_de

        print(f"   Algorithm   : {algo}")
        print(f"   Public Key  : {len(public_key)} bytes")
        print(f"   Signature   : {len(signature)} bytes")
        print(f"   Valid       : {'YES' if is_valid else 'NO'}")
        print()
        print(f"{'Algorithm':<25} {'KeyGen':>10} {'Signing':>10} {'Verification':>10} {'Total':>10}")
        print("-" * 65)
        print(f"{algo:<25} {avg_kg:>9.3f}ms {avg_en:>9.3f}ms {avg_de:>9.3f}ms {total:>9.3f}ms")
        print()
        print()
            
    except Exception as e:
        print(f"  {algo} not available: {e}\n")

   Algorithm   : SPHINCS+-SHA2-128f-simple
   Public Key  : 32 bytes
   Signature   : 17088 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
SPHINCS+-SHA2-128f-simple     6.967ms   162.644ms     9.476ms   179.087ms


   Algorithm   : SPHINCS+-SHA2-192f-simple
   Public Key  : 48 bytes
   Signature   : 35664 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
SPHINCS+-SHA2-192f-simple    16.614ms   405.115ms    22.228ms   443.957ms


   Algorithm   : SPHINCS+-SHA2-256f-simple
   Public Key  : 64 bytes
   Signature   : 49856 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
SPHINCS+-SHA2-256f-simple    40.676ms   904.151ms    35.929ms   980.755ms


In [14]:
#Tamper Detection Test

with oqs.Signature("SPHINCS+-SHA2-256f-simple") as signer:
    public_key = signer.generate_keypair()

    original_message = b"Original Secure Message!!!"
    tampered_message = b"Tampered Secure Message!!!"

    #Sign original
    signature = signer.sign(original_message)

    #Verify Original
    valid_original = signer.verify(original_message, signature, public_key)

    #Verify Tampered
    valid_tampered = signer.verify(tampered_message, signature, public_key)

    print("   Tamper Detection Test")
    print(f"   Original Message Verified : {'PASS' if valid_original else 'FAIL'}")
    print(f"   Tampered Message Verified : {'PASS' if valid_tampered else 'FAIL'}")
    print()
    print("Tamper was detected!" if not valid_tampered else "Tamper NOT detected!")

   Tamper Detection Test
   Original Message Verified : PASS
   Tampered Message Verified : FAIL

Tamper was detected!


## Falcon

In [15]:
#Generate Falcon keys
ALGORITHM = "Falcon-1024"

with oqs.Signature(ALGORITHM) as signer_setup:
    public_key = signer_setup.generate_keypair()

    secret_key = signer_setup.export_secret_key()
    
print(f"\n Algorithm            : {ALGORITHM}")
print(f" Keypair Generated!")
print(f"\n    Verification Key  : {public_key.hex()[:40]}...")
print(f"    Signing Key       : {secret_key.hex()[:40]}...")
print(f"\n    Verification Key Size : {len(public_key)} bytes")
print(f"    Signing Key Size      : {len(secret_key)} bytes")
print("\n Setup Complete!\n")


 Algorithm            : Falcon-1024
 Keypair Generated!

    Verification Key  : 0a39983590dbcc03b3cec889c0c15f23180f676f...
    Signing Key       : 5a06c39f6fc30901ecffff1043d0042210082f73...

    Verification Key Size : 1793 bytes
    Signing Key Size      : 2305 bytes

 Setup Complete!



In [16]:
#Sign a message
message = b"I am securing this message with Falcon keys!!!"

with oqs.Signature(ALGORITHM, secret_key) as signer:
    signature = signer.sign(message)

print(f"\n  Signature Generated!")
print(f"   Signature            : {signature.hex()[:40]}...")
print(f"   Signature Size       : {len(signature)} bytes")
print("\n Signing Complete!\n")


  Signature Generated!
   Signature            : 3a669e3a25c85870c97e63c14be2dc88ea106103...
   Signature Size       : 1266 bytes

 Signing Complete!



In [17]:
#Verify the message
with oqs.Signature(ALGORITHM) as verifier:
    is_valid = verifier.verify(
        message,
        signature,
        public_key
    )

print(f"\n Verifying Signature...")
print(f"   Verified with        : Verification Key (Public)")
print(f"   Result               : {' VALID' if is_valid else ' INVALID'}")

if not is_valid:
    raise Exception(" Signature verification failed!")


 Verifying Signature...
   Verified with        : Verification Key (Public)
   Result               :  VALID


In [18]:
print("=" * 60)
print("         PHASE 4 — TAMPER DETECTION TESTS")
print("=" * 60)

# ── Test 1: Tampered Message ───────────────────────────────────
tampered_message = b"This message was TAMPERED by an attacker!"

with oqs.Signature(ALGORITHM) as verifier:
    test1 = verifier.verify(tampered_message, signature, verification_key)

print(f"\n    Test 1 — Tampered Message")
print(f"      Original  : {message.decode()[:45]}...")
print(f"      Tampered  : {tampered_message.decode()}")
print(f"      Valid     : {test1}")
print(f"      {' Rejected!' if not test1 else ' Accepted — unexpected!'}")

# ── Test 2: Forged Signature ───────────────────────────────────
forged_sig = os.urandom(len(signature))

with oqs.Signature(ALGORITHM) as verifier:
    test2 = verifier.verify(message, forged_sig, verification_key)

print(f"\n    Test 2 — Forged Signature")
print(f"      Forged Sig : {forged_sig.hex()[:40]}...")
print(f"      Valid      : {test2}")
print(f"      {' Rejected!' if not test2 else ' Accepted — unexpected!'}")

# ── Test 3: Wrong Verification Key ────────────────────────────
with oqs.Signature(ALGORITHM) as fake_setup:
    fake_vk = fake_setup.generate_keypair()

with oqs.Signature(ALGORITHM) as verifier:
    test3 = verifier.verify(message, signature, fake_vk)

print(f"\n    Test 3 — Wrong Verification Key")
print(f"      Fake VK    : {fake_vk.hex()[:40]}...")
print(f"      Valid      : {test3}")
print(f"      {' Rejected!' if not test3 else ' Accepted — unexpected!'}")

print("\n Tamper Detection Tests Complete!\n")


         PHASE 4 — TAMPER DETECTION TESTS

    Test 1 — Tampered Message
      Original  : I am securing this message with Falcon keys!!...
      Tampered  : This message was TAMPERED by an attacker!
      Valid     : False
       Rejected!

    Test 2 — Forged Signature
      Forged Sig : 8d7077b972b05fad4800266337bc0d54c72e2e13...
      Valid      : False
       Rejected!

    Test 3 — Wrong Verification Key
      Fake VK    : 0a13a2797612a69a207c12eb945cab6820d78163...
      Valid      : False
       Rejected!

 Tamper Detection Tests Complete!



In [19]:
#Digital Signature - Sign and Verify Demo

sig_algorithms = ['Falcon-512', 'Falcon-1024']

keygen_times = []
sign_times = []
verify_times = []
ITERATIONS = 1000

message = b"Hello Post Quantum World! This message is quantum safe-signed!"

for algo in sig_algorithms:
    try:
        for _ in range(ITERATIONS):
            with oqs.Signature(algo) as signer:
                #Generate Key Pair
                t0 = time.perf_counter()
                public_key = signer.generate_keypair()
                keygen_times.append(time.perf_counter() - t0)
                
    
                #Sign
                t0 = time.perf_counter()
                signature = signer.sign(message)
                sign_times.append(time.perf_counter() - t0)
    
                #Verify
                t0 = time.perf_counter()
                is_valid = signer.verify(message, signature, public_key)
                verify_times.append(time.perf_counter() - t0)
    
        avg_kg = sum(keygen_times) / ITERATIONS * 1000
        avg_en = sum(sign_times)  / ITERATIONS * 1000
        avg_de = sum(verify_times)  / ITERATIONS * 1000
        total  = avg_kg + avg_en + avg_de

        print(f"   Algorithm   : {algo}")
        print(f"   Public Key  : {len(public_key)} bytes")
        print(f"   Signature   : {len(signature)} bytes")
        print(f"   Valid       : {'YES' if is_valid else 'NO'}")
        print()
        print(f"{'Algorithm':<25} {'KeyGen':>10} {'Signing':>10} {'Verification':>10} {'Total':>10}")
        print("-" * 65)
        print(f"{algo:<25} {avg_kg:>9.3f}ms {avg_en:>9.3f}ms {avg_de:>9.3f}ms {total:>9.3f}ms")
        print()
        print()
            
    except Exception as e:
        print(f"  {algo} not available: {e}\n")

   Algorithm   : Falcon-512
   Public Key  : 897 bytes
   Signature   : 656 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
Falcon-512                   31.621ms     2.334ms     0.244ms    34.200ms


   Algorithm   : Falcon-1024
   Public Key  : 1793 bytes
   Signature   : 1270 bytes
   Valid       : YES

Algorithm                     KeyGen    Signing Verification      Total
-----------------------------------------------------------------
Falcon-1024                 108.901ms     6.860ms     0.690ms   116.450ms


